# Generate CelebDFv1 Augmentation Test Embeddings

Generate CLIP embeddings cho CelebDFv1 với test-time image augmentations.

Output lưu đủ:

- `features_views`: `[N, 4, D]` embeddings của từng augmented view
- `features`: `[N, D]` mean-normalized embedding để tương thích pipeline cũ

## Kaggle Setup

In [ ]:
!git clone -b dev https://github.com/hoavien0110/training-free-tta-for-deepfake-detection.git /kaggle/working/training-free-tta-for-deepfake-detection
%cd /kaggle/working/training-free-tta-for-deepfake-detection
!pip install -q open-clip-torch

## Check Inputs

Nếu Kaggle slug khác, sửa path trong cell Run. `CELEB_CORR_ROOT` phải là folder ảnh corruption, không phải folder `.pt` embeddings.

In [ ]:
!ls -lah /kaggle/input
!find /kaggle/input/datasets/jamestashvik/deepfakebench -maxdepth 2 -type f -name "*.csv" -print
!find /kaggle/input -maxdepth 3 -type d \( -name "Celeb-real" -o -name "YouTube-real" -o -name "Celeb-synthesis" \) | sort | sed -n "1,120p"

## Run

Mặc định gen clean CelebDFv1 + 4 corruption level 2, mỗi sample có `4` augmented views. Notebook này lưu `features_views` mặc định.

In [ ]:
DEEPFAKEBENCH = "/kaggle/input/datasets/jamestashvik/deepfakebench"
CELEB_CORR_ROOT = "/kaggle/input/celebdfv1-corruption-level-2"

!python training/generate_celebdfv1_augmentation_test_embeddings.py \
  --include-clean \
  --level 2 \
  --csv-path {DEEPFAKEBENCH}/deepfakebench_dataset.csv \
  --deepfakebench-root {DEEPFAKEBENCH}/DeepFakeBench \
  --corruption-root {CELEB_CORR_ROOT} \
  --corruptions color_contrast color_saturation gaussian_blur resize \
  --augmentation-policy weak \
  --views 4 \
  --output-dir /kaggle/working/celebdfv1_augmented_test_features \
  --batch-size 16 \
  --num-workers 4 \
  --device auto \
  --use-amp \
  --trust-paths \
  --skip-existing

## Preview Outputs

In [ ]:
!find /kaggle/working/celebdfv1_augmented_test_features -maxdepth 1 -type f -name "*.pt" -print | sort

In [ ]:
import torch
from pathlib import Path

path = sorted(Path("/kaggle/working/celebdfv1_augmented_test_features").glob("*.pt"))[0]
payload = torch.load(path, map_location="cpu")
print(path)
print("features", tuple(payload["features"].shape))
print("labels", tuple(payload["labels"].shape))
print("features_views", None if "features_views" not in payload else tuple(payload["features_views"].shape))
print("augmentation_policy", payload.get("augmentation_policy"))
print("augmentation_views", payload.get("augmentation_views"))
print("counts", torch.bincount(payload["labels"].long(), minlength=2).tolist())

## Level 5 Variant

Nếu level 5 nằm ở các Kaggle dataset riêng, dùng map từng corruption dưới đây và sửa path cho đúng mount.

In [ ]:
# !python training/generate_celebdfv1_augmentation_test_embeddings.py \
#   --level 5 \
#   --csv-path {DEEPFAKEBENCH}/deepfakebench_dataset.csv \
#   --deepfakebench-root {DEEPFAKEBENCH}/DeepFakeBench \
#   --corruption-root-map \
#     color_contrast=/kaggle/input/celebdfv1-color-contrast-5 \
#     color_saturation=/kaggle/input/celebdfv1-color-saturation-5 \
#     gaussian_blur=/kaggle/input/celebdfv1-gaussian-blur-5 \
#     resize=/kaggle/input/celebdfv1-resize-5 \
#   --corruptions color_contrast color_saturation gaussian_blur resize \
#   --augmentation-policy weak \
#   --views 4 \
#   --output-dir /kaggle/working/celebdfv1_level5_augmented_test_features \
#   --batch-size 16 \
#   --num-workers 4 \
#   --device auto \
#   --use-amp \
#   --trust-paths \
#   --skip-existing